# EDA: Цены на дома

Этот ноутбук показывает компактный, но осмысленный EDA для соревнования `House Prices - Advanced Regression Techniques`.
Здесь мы проверяем форму таргета, характер пропусков и связь числовых признаков с ценой, чтобы затем осознанно выбирать признаки и модели.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

# Notebook works both from project root and from notebooks/
candidate_roots = [Path.cwd(), Path.cwd().parent]
project_root = next((p for p in candidate_roots if (p / "data" / "train.csv").exists()), None)
if project_root is None:
    raise FileNotFoundError(
        "Не найден data/train.csv. Откройте ноутбук из папки проекта или проверьте структуру data/."
    )

data_dir = project_root / "data"
train = pd.read_csv(data_dir / "train.csv")
test = pd.read_csv(data_dir / "test.csv")

train["SalePriceLog"] = np.log1p(train["SalePrice"])

print(f"Текущая рабочая папка: {Path.cwd()}")
print(f"Используемая папка данных: {data_dir}")
display(train.head())
print(f"Форма train: {train.shape}")
print(f"Форма test: {test.shape}")
print(f"Средняя цена продажи: {train['SalePrice'].mean():.0f}")
print(f"Медиана цены продажи: {train['SalePrice'].median():.0f}")
print(f"Среднее log1p(SalePrice): {train['SalePriceLog'].mean():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(train["SalePrice"], bins=40, kde=True, ax=axes[0], color="#2a6f97")
axes[0].axvline(train["SalePrice"].median(), color="black", linestyle="--", linewidth=1, label="median")
axes[0].axvline(train["SalePrice"].mean(), color="red", linestyle=":", linewidth=1, label="mean")
axes[0].set_title("Распределение SalePrice")
axes[0].set_xlabel("Цена продажи")
axes[0].legend()

sns.histplot(train["SalePriceLog"], bins=40, kde=True, ax=axes[1], color="#ff7f0e")
axes[1].axvline(train["SalePriceLog"].median(), color="black", linestyle="--", linewidth=1, label="median")
axes[1].axvline(train["SalePriceLog"].mean(), color="red", linestyle=":", linewidth=1, label="mean")
axes[1].set_title("Распределение log1p(SalePrice)")
axes[1].set_xlabel("log1p(Цена продажи)")
axes[1].legend()

plt.tight_layout()

print("Идея: log1p сжимает правый хвост и делает таргет более симметричным.")
print("Это полезно для линейных моделей и для стабильной оценки ошибки на дорогих домах.")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

# 1) Самый сильный числовой сигнал: OverallQual
overall_qual = train.groupby("OverallQual", as_index=False)["SalePrice"].median().sort_values("OverallQual")
sns.barplot(data=overall_qual, x="OverallQual", y="SalePrice", ax=axes[0, 0], color="#4c78a8")
axes[0, 0].set_title("Медианная цена по OverallQual")
axes[0, 0].set_xlabel("OverallQual")
axes[0, 0].set_ylabel("Медианная SalePrice")

# 2) Район как сильный категориальный признак
neighborhood_stats = train.groupby("Neighborhood", as_index=False)["SalePrice"].agg(["median", "count"]).reset_index()
neighborhood_top = neighborhood_stats.query("count >= 20").sort_values("median", ascending=False).head(10)
sns.barplot(data=neighborhood_top, y="Neighborhood", x="median", ax=axes[0, 1], color="#7aa6c2")
axes[0, 1].set_title("Топ районов по медианной цене (count >= 20)")
axes[0, 1].set_xlabel("Медианная SalePrice")
axes[0, 1].set_ylabel("Neighborhood")

# 3) Площадь жилой части: сильная связь, но с выбросами
sns.scatterplot(data=train, x="GrLivArea", y="SalePrice", ax=axes[1, 0], alpha=0.5, color="#d97706")
axes[1, 0].set_title("SalePrice vs GrLivArea")
axes[1, 0].set_xlabel("GrLivArea")
axes[1, 0].set_ylabel("SalePrice")

# 4) Пропуски и числовые связи с log-таргетом
missing = train.isna().mean().sort_values(ascending=False).head(12)
num_cols = train.select_dtypes(include=["number"]).columns
corr = train[num_cols].corr(numeric_only=True)["SalePriceLog"].sort_values(ascending=False)
strong_corr = corr.drop("SalePriceLog").head(10)

sns.barplot(x=missing.values, y=missing.index, ax=axes[1, 1], color="#64748b")
axes[1, 1].set_title("Топ столбцов по доле пропусков")
axes[1, 1].set_xlabel("Доля пропусков")
axes[1, 1].set_ylabel("Столбец")

plt.tight_layout()

print("Самые сильные числовые связи с log1p(SalePrice):")
display(strong_corr.rename("corr_with_log_price").to_frame())
print("Сильные категории, которые полезно кодировать отдельно:")
display(neighborhood_top[["Neighborhood", "median", "count"]].rename(columns={"median": "median_saleprice"}))
print("Примеры возможных выбросов по GrLivArea: очень большая площадь, но цена не растет пропорционально.")
display(train.sort_values("GrLivArea", ascending=False)[["GrLivArea", "SalePrice", "OverallQual", "Neighborhood"]].head(8))

## Короткие выводы

- `SalePrice` сильно скошен вправо, поэтому в пайплайне удобно использовать `log1p(SalePrice)` — так таргет становится более стабильным для обучения.
- Самые сильные простые сигналы здесь — `OverallQual`, `GrLivArea`, район (`Neighborhood`) и часть числовых размеров дома.
- На ценe заметно отражаются и категориальные признаки, особенно район и качество дома, поэтому их важно кодировать аккуратно, а не оставлять как есть.
- Пропуски распределены неравномерно: часть колонок лучше заполнять осознанно, а не просто удалять.

Этот ноутбук специально компактный, но он уже показывает: где таргет перекошен, какие признаки самые сильные и почему `log1p` — разумный старт для регрессии по цене дома.